In [ ]:
import torch
import triton
import triton.language as tl
import time


# vector sum

numpy implementation

In [ ]:
def vector_add_pseudocode(a, b, output, grid):

    PIDS = torch.arange(grid[0])

    a_ptr = 0
    b_ptr = 0
    output_ptr = 0

    for pid in PIDS:    # parallel
        a_ptr_offset = a_ptr + pid
        b_ptr_offset = b_ptr + pid
        output_ptr_offset = output_ptr + pid

        output[output_ptr_offset] = a[a_ptr_offset] + b[b_ptr_offset]


def vector_add_pseudocode_wrapper(a, b):

    n_elements = len(a)
    grid = (n_elements,)

    output = torch.empty_like(a)

    vector_add_pseudocode(a, b, output, grid)

    return output

In [ ]:
a = torch.rand(10, )
b = torch.rand(10, )

print(vector_add_pseudocode_wrapper(1, b) == a + b)

triton kernel

In [ ]:
@triton.jit
def vector_add(a_ptr, b_ptr, output_ptr):

    pid = tl.program_id(axis=0)

    a_ptr_offset = a_ptr + pid
    b_ptr_offset = b_ptr + pid
    output_ptr_offset = output_ptr + pid

    a = tl.load(a_ptr_offset)
    b = tl.load(b_ptr_offset)

    output = a + b

    tl.store(output_ptr_offset, output)



In [ ]:
def vector_add_wrapper(a, b):

    output = torch.empty_like(a)

    n_elements = len(a)
    grid = (n_elements, )


    vector_add[grid](a, b, output)

    return output



In [ ]:
def benchmark_torch(a, b, iters=100):

    # warmup
    for _ in range(10):
        _ = a + b
    torch.cuda.synchronize()

    start = time.time()
    for _ in range(iters):
        out = a + b

    torch.cuda.synchronize()
    end = time.time()

    return (end - start) / iters

In [ ]:
def benchmark_triton(a, b, iters=100):

    # warmup
    for _ in range(10):
        _  = vector_add_wrapper(a, b)
    torch.cuda.synchronize()

    start = time.time()
    for _ in range(iters):
        out = vector_add_wrapper(a, b)
    torch.cuda.synchronize()
    end = time.time()

    return (end - start) / iters

In [ ]:
if __name__ == "__main__":
    a = torch.randn(1000).to("cuda")
    b = torch.randn(1000).to("cuda")

    psuedo_output = vector_add_pseudocode_wrapper(a, b)
    triton_output = vector_add_pseudocode(a, b)
    torch_output = a + b

    assert torch.allclose(psuedo_output, triton_output)
    assert torch.allclose(triton_output, torch_output)

    print("triton matches torch!!")

    # speed check
    N = 50_000_000
    a = torch.randn(N, device="cuda")
    b = torch.randn(N, device="cuda")

    # becnhmark
    torch_time = benchmark_torch(a, b)
    triton_time = benchmark_triton(a, b)

    print(f'torch time per iter: {torch_time * 1e3:.3f} ms')
    print(f'triton time per iter: {triton_time * 1e3:.3f} ms')

# vector sum blocked

In [ ]:
import time
import math
import torch
import triton
import triton.language as tl

In [ ]:
def blocked_voctor_add_pseudocode(a, b, output, grid, n_elements, BLOCK_SIZE):

    PIDS = torch.arange(grid[0])

    a_ptr = 0
    b_ptr = 0
    output_ptr = 0

    for pid in PIDS:

        block_start = pid * BLOCK_SIZE
        block_indexes = block_start + torch.arange(BLOCK_SIZE)

        mask = (block_indexes < n_elements)
        block_indexes = block_indexes[mask]

        a_ptr_offsets = a_ptr + block_indexes
        b_ptr_offsets = b_ptr + block_indexes
        output_ptr_offsets = output_ptr + block_indexes

        output[output_ptr_offsets] = a[a_ptr_offsets] + b[b_ptr_offsets]



In [ ]:
def blocked_vector_add_pseudocode_wrapper(a, b):

    BLOCK_SIZE = 1024
    n_elements = len(a)
    grid = (tl.cdiv(n_elements, BLOCK_SIZE), )

    output = torch.empty_like(a)

    blocked_voctor_add_pseudocode(a, b, output, grid, n_elements, BLOCK_SIZE)

    return output

In [ ]:
@triton.jit
def blocked_vector_add(a_ptr,
                       b_ptr,
                       output_ptr,
                       n_elements,
                       BLOCK_SIZE: tl.constexpr):

    pid = tl.program_id(axis=0)

    block_start = pid * BLOCK_SIZE
    offsets = block_start + tl.arange(0, BLOCK_SIZE)

    mask = (offsets < n_elements)

    a = tl.load(a_ptr + offsets, mask=mask)
    b = tl.load(b_ptr + offsets, mask=mask)

    output = a + b

    tl.store(output_ptr + offsets, output, mask=mask)




In [ ]:
def blocked_vector_add_wrapper(a, b)

    output = torch.empty_like(a)

    n_elements = len(a)
    grid = lambda args: (triton.cdiv(n_elements, args["BLOCK_SIZE"]), )

    blocked_vector_add[grid](a, b, output, n_elements, BLOCK_SIZE=1024)

    return output